### **ACID - PART 6 - Extract feature**

**Author:** Alessandro Ulivi (alessandro.ulivi.89@gmail.com)

**Last update (yyyy/mm/dd):** 2026/08/20

## **---------------------------**

### Import required modules

Run the following cell.

Don't modify the following cell.

In [1]:
# Import required modules
from importlib.metadata import version
import datetime
import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter
import napari
import tifffile
from skimage.transform import resize
from skimage.measure import regionprops_table
# import tifffile
# import matplotlib.pyplot as plt
from acid.utils.listdirNHF import listdirNHF
from acid.utils.get_defaults import default_file_name
from acid.utils.label_image_utils import exclude_label_on_edge
from acid.feature_extraction.default_regionprops import default_regionpros_props
from acid.feature_extraction.extra_regionprops import regionpros_extra_props
from acid.feature_extraction.measure_hessian_matrix import MeasureHessianMatrix
from acid.feature_extraction.measure_structure_tensor import MeasureStructureTensor



### Specify the paths to input and output data directories - specify hyperparameters

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modified are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [ ]:
# indicate the path to the directory storing the input images
fov_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\fov_proc"

# indicate the path to the directory storing the segmentations
segmentation_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\seg"

# indicate the path to the directory where outputs will be saved
output_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\measurement"

# metadata_df directory - indicate the full path to the directory where part1 metadata have been saved
metadata_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\proc_metadata"

# name of the metadata file - NOTE: this is expected to be the metadata_df saved
# from part4b notebook
# the following options are possible:
# 1) write the full name (extension included) of the csv file
# 2) write "default" or any other version with a uppercase letter (e.g. "Default")
# 3) leave an empty string: ""
# 4) write None (NOTE that this is not a string, there are no "", aka it is not "None")
# Options 2,3 and 4 will lead to the same behavior: the most recently saved csv file will be used.
# By default, the timestamp of the last file modification is used. As an alternative it is possible to use the
# date saved in the name (change below parameter metadata_from_file_name to True -
# see utils.get_defaults.default_file_name documentation).
metadata_file_name = "20260320_ACID_metadata_part_4b.csv" # this is a placeholder


# --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---
# # --- for image preprocessing ---
# indicate the channel axis position - image preprocessing is iterated over the individual channels
channel_axis = 0

# indicate the sigma of gaussian smoothing
sigma = 3

# # --- parameters used for importing the default metadata dataframe ---
# Indicate the part of the file name to use to select files into metadata_directory to be used for selecting the default
# file
default_metadata_file_target = ".csv" # files with this string in their name will be selected for the default file selection - if None, all files will be selected
default_metadata_file_exclude = None # files with this string in their name will be excluded from the default file selection - if None, no files will be excluded

# Indicate whether to extract the date information from the file name when selecting the default file
# If False, the date will be extracted from the file's last modified timestamp, if True, from the file name.
# If from_file_name is True, the parameters *_default_separator and *_default_date_position
# should be used to indicate, respectively the separator to use for splitting the file name into tokens and
# the position of the token containing the date information. In addition, the date format used to include the date in the file name
# should be indicated using the parameters *_default_date_format.
# For example, if metadata_from_file_name is True, and the file name is "251126_plate_layout.csv",
# "_" is used as separator, 0 as date position, and '%Y%m%d' as date format.
# Ref to utils.get_defaults.default_file_name for more details.
metadata_from_file_name = False

# separator - used to split the file name and extract the information token with the date
# if metadata_from_file_name is True
metadata_default_separator = '_'

# date position - the position of the date information token after splitting the file name using file_name_separator (above)
# used if metadata_from_file_name is True
metadata_default_date_position = 0

# date format - the format used for including the date in the file name to be opened by default
# used if metadata_from_file_name is True
metadata_default_date_format='%Y%m%d'

# reverse - if True, the file with the most recent date will be returned by default - if False, the opposite
metadata_default_reverse = True

# --- parameters for updating metadata data frame ---
# image segmentation metadata dataframe - computation date format - this is the format to be used for
# indicating the date when the image segmentation was done. This is used for saving the date in the
# metadata dataframe
metadata_df_meta_date_format = '%y%m%d'

# indicate preprocessing steps
preprocessing_steps = f"gaussian smoothing individual channels with sigma {sigma}"

metadata_df_date_clm_name = "features_extraction_date"
metadata_df_file_name_clm_name = "features_data_frame"
metadata_df_method_clm_name = "features_extraction_preprocessing"


# --- parameters for file saving ---
# separator used for saved file names
save_file_name_separator = '_'

# project
project_name = "ACID"

# processing metadata dataframe name - savingword
metadata_savingword = "metadata"

# processing metadata dataframe name - file suffix
metadata_file_suffix = f"part{save_file_name_separator}6.csv"

# processing metadata dataframe name - date format
metadata_date_format = '%Y%m%d'

# hyperparameters dataframe name - date format
hyperparameters_date_format = '%Y%m%d-%H%M%S'

# hyperparameters dataframe name - savingword
hyperparameters_savingword = "hyperparameters"

# hyperparameters dataframe name - file suffix
hyperparameters_file_suffix = f"part{save_file_name_separator}4extra.csv"

save_csv_index = False


# --- parameters for saving secondary information ---
# indicate the name of the directory for saving secondary outputs -
# this directory is used to store the hyperparameters used per each run of the pipeline
secondary_output_directory = "secondary_output"
exist_ok = True # if the secondary output directory already exists, do not raise an error


### Create output directory and secondary output directory if they don't exist

##### Output directory stores the segmentation masks
##### Secondary output directory is used to store the hyperparameters used per each run of the pipeline

Run the following cell.

Don't modify the following cell.

In [3]:
# create the output_directory if it doesn't exist
if not os.path.exists(output_directory):
    os.makedirs(output_directory, exist_ok=exist_ok)

# create the path to secondary_output directory
secondary_output_path = os.path.join(os.getcwd(), secondary_output_directory)

# create the secondary_output directory if it doesn't exist
if not os.path.exists(secondary_output_path):
    os.makedirs(secondary_output_path, exist_ok=exist_ok)
    

#### Open the metadata dataframe - this is expected to be the output of part 5

Run the following cell.

Don't modify the following cell.

In [4]:
# check if using the default metadata data frame (the most recently saved)
if metadata_file_name==None or metadata_file_name.lower()=="default" or metadata_file_name=="":
    
    # import target files in the metadata_directory
    metadata_files = listdirNHF(metadata_directory,
                                target=default_metadata_file_target,
                                exclude=default_metadata_file_exclude)
    
    # get the default metadata file
    metadata_file_name = default_file_name(file_list=metadata_files,
                                           from_file_name=metadata_from_file_name,
                                           directory_path=metadata_directory,
                                           separator=metadata_default_separator,
                                           date_position=metadata_default_date_position,
                                           date_format=metadata_default_date_format,
                                           reverse=metadata_default_reverse)
    
    print(f"using {metadata_file_name} as default metadata file")


# open the metadata file
metadata_df_i = pd.read_csv(os.path.join(metadata_directory, metadata_file_name))

# copy metadata_df
metadata_df = metadata_df_i.copy()

# # display the metadata dataframe
# metadata_df


In [5]:
for clm in metadata_df.columns:
    print(clm)

raw_file_name
scene_name
processing_date_yymmdd
ome_tif_file_name
location
microscope
objective
experiment
condition1
infectious_organism
condition_2
imaging_hours
well
channel_0
channel_1
channel_2
channel_3
channel_4
physical_size_unit_x
physical_size_unit_y
dtype
size_t
size_c
size_z
size_y
physical_size_y
size_x
physical_size_x
dims_order
int_well
treatment
is_train
plls-0
plls-1
plls-2
plls-3
plls-4
bottom_percentile_fraction-0
bottom_percentile_fraction-1
bottom_percentile_fraction-2
bottom_percentile_fraction-3
bottom_percentile_fraction-4
top_percentile_fraction-0
top_percentile_fraction-1
top_percentile_fraction-2
top_percentile_fraction-3
top_percentile_fraction-4
mean_over_std-0
mean_over_std-1
mean_over_std-2
mean_over_std-3
mean_over_std-4
flag
background_funct_date
background_funct_avg_method
background_funct_ball_radius
background_funct_white_bg-0
background_funct_white_bg-1
background_funct_white_bg-2
background_funct_white_bg-3
background_funct_white_bg-4
background_fu

### MAIN LOOP
#### 3.1. Preprocess image and segmentation mask
#### 3.2. Extract features
#### 3.3. Save results 

## **--- --- ---**

Run the following cell.

Don't modify the following cell.

In [ ]:
# initialize lists to collect processing metadata for updating - this will be used to update the metadata df
features_extraction_date_collection = []
features_file_name_collection = []
preprocessing_steps = []


# get properties to measure
properties = default_regionpros_props()

# get extra properties to measure
extra_properties = regionpros_extra_props()


# iterate through the rows of the metadata dataframe
for file_idx in metadata_df.index:

    print("---------    ---------")
    
    # ---------
    # OPEN FIELD OF VIEW AND SEGMENTATION
    # ---------
    try:
        # get the name of the field of view as ome.tif file
        # field_of_view_file = metadata_df.loc[file_idx, illum_correct_df_file_name_clm_name]
        field_of_view_file = metadata_df.loc[file_idx, 'illumination_correction_file_name']

        # form the full path to the field of view file
        fov_path = os.path.join(fov_directory,field_of_view_file)

        # open the field of view image as an array
        field_of_view = tifffile.imread(fov_path)
    
        print(f"working on {field_of_view_file}")
        
    except:
        print(f"can't open {field_of_view_file}, skipping image segmentation")
        continue

    try:
        # get the name of the segmentation mask
        segmentation_file =  f"{field_of_view_file.removesuffix(".ome.tif")}.tif" # this is a placeholder
        # segmentation_file = metadata_df.loc[file_idx, '']

        # form the full path to the segmentation mask
        segmentation_path = os.path.join(segmentation_directory, segmentation_file)
            
        # open the segmentation mask as an array
        segmentation = tifffile.imread(segmentation_path)

        print(f"fetched segmentation file: {segmentation_file}")
            
    except:
        print(f"can't open segmentation mask, skipping image segmentation")
        continue
    

    # ---------
    # PREPROCESSING FIELD OF VIEW
    # ---------
    # move channel axis to position -1, to make next steps invariant
    # to channel axis position and for compatibility with skimage.measure.regionprops
    preprocessed = np.moveaxis(field_of_view, channel_axis, -1)

    # smooth field of view along channel axis - to get rid of some noise
    preprocessed = gaussian_filter(preprocessed, sigma=sigma, axes=-1) # note: channel_axis is know to be in position -1

    # ---------
    # PREPROCESSING SEGMENTATION MASK
    # ---------
    # remove objects touching the boarder
    label_image = exclude_label_on_edge(segmentation)

    # ---------
    # MEASUREMENT EXTRACTION
    # ---------
    # extract measurements for the nucleus and cytosol segmentations
    regionprops_features = pd.DataFrame(regionprops_table(label_image,
                                              intensity_image=preprocessed,
                                              properties=properties,
                                              extra_properties=extra_properties))
    print(regionprops_features.shape)
    # istantiate a hessian matrix measurer
    hessian_measurer = MeasureHessianMatrix(preprocessed)
    hessian_features = hessian_measurer.measure_obj_hessian_matrix_eigenval(label_image=label_image,
                                                                            axis=-1)
    print(hessian_features.shape)
    # istantiate a structure tensor measurer
    structure_tensor_measurer = MeasureStructureTensor(preprocessed)
    structure_tensor_features = structure_tensor_measurer.measure_obj_struct_tensor_eigenval(label_image=label_image,
                                                                                             axis=-1)

    print(structure_tensor_features.shape)
    # ---------
    # MERGE MEASUREMENT
    # ---------
    merged_features = regionprops_features.merge(hessian_features, on="label", how="right")
    print(merged_features.shape)
    merged_features = merged_features.merge(structure_tensor_features, on="label", how="right")
    print(merged_features.shape)

    # print a update
    print("features measured")

    # ---------
    # SAVE THE RESULTS
    # ---------
    feature_save_name = f"{field_of_view_file.removesuffix(".ome.tif")}.csv"
    merged_features.to_csv(os.path.join(output_directory, feature_save_name), index=save_csv_index)

    # print a update
    print("features saved")

    # ---------   ---------
    # UPDATE METADATA DATAFRAME
    # ---------   ---------
    features_extraction_date_collection.append(datetime.datetime.now().strftime(metadata_df_meta_date_format))
    features_file_name_collection.append(feature_save_name)
    preprocessing_steps.append(preprocessing_steps)

    
print("")
print("Features extraction completed")

# ---------   ---------
# UDDATE METADATA DICTIONARY
# ---------   ---------
metadata_df[metadata_df_date_clm_name] = features_extraction_date_collection
metadata_df[metadata_df_file_name_clm_name] = features_file_name_collection
metadata_df[metadata_df_method_clm_name] = preprocessing_steps


# ---------   ---------
# SAVE THE FINAL METADATA FILE
# ---------   ---------

# save the updated metadata dataframe as a csv file
metadata_df_name = f"{datetime.datetime.now().strftime(metadata_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{metadata_savingword}{save_file_name_separator}{metadata_file_suffix}"
metadata_df.to_csv(os.path.join(metadata_directory, metadata_df_name), index=save_csv_index)

# print progress update
print("processing metadata saved")
print("finished")

---------    ---------
working on H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A1_bg.ome.tif
can't open segmentation mask, skipping image segmentation
---------    ---------
working on H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A2_bg.ome.tif
can't open segmentation mask, skipping image segmentation
---------    ---------
working on H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A5_bg.ome.tif
can't open segmentation mask, skipping image segmentation
---------    ---------
working on H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A6_bg.ome.tif
can't open segmentation mask, skipping image segmentation
---------    ---------
working on H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_B7_bg.ome.tif
can't open segmentation mask, skipping image segmentation
---------    ---------
working on H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_B6_bg.ome.tif
can't open segmentation mask, skipping image segmentation
---------    ---------
working on H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_B5_bg.ome.tif
can't op

### Save analysis hyperparameters

Run the following cell.

Don't modify the following cell.

In [ ]:
# # collect hyperparameters in a dictionary

hyperparameter_dict = {

'fov_directory':fov_directory,
'segmentation_directory':segmentation_directory,
'output_directory':output_directory,
'metadata_directory':metadata_directory,
'metadata_file_name':metadata_file_name,
'channel_axis':channel_axis,
'sigma':sigma,
'default_metadata_file_target':default_metadata_file_target,
'default_metadata_file_exclude':default_metadata_file_exclude,
'metadata_from_file_name': metadata_from_file_name,
'metadata_default_separator':metadata_default_separator,
'metadata_default_date_position':metadata_default_date_position,
'metadata_default_date_format':metadata_default_date_format,
'metadata_default_reverse':metadata_default_reverse,
'metadata_df_meta_date_format':metadata_df_meta_date_format,
'preprocessing_steps':preprocessing_steps,
'metadata_df_date_clm_name':metadata_df_date_clm_name,
'metadata_df_file_name_clm_name':metadata_df_file_name_clm_name,
'metadata_df_method_clm_name':metadata_df_method_clm_name,
'save_file_name_separator':save_file_name_separator,
'project_name':project_name,
'metadata_savingword':metadata_savingword,
'metadata_file_suffix':metadata_file_suffix,
'metadata_date_format':metadata_date_format,
'hyperparameters_date_format':hyperparameters_date_format,
'hyperparameters_savingword':hyperparameters_savingword,
'hyperparameters_file_suffix':hyperparameters_file_suffix,
'secondary_output_directory':secondary_output_directory,
'exist_ok':exist_ok

}



# transform the hyperparameter_dict in a pandas series
hyperparameter_series = pd.Series(hyperparameter_dict)

# save hyperparamters
hyperparameter_saving_name = f"{datetime.datetime.now().strftime(hyperparameters_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{hyperparameters_savingword}{save_file_name_separator}{hyperparameters_file_suffix}"
hyperparameter_series.to_csv(os.path.join(secondary_output_directory,hyperparameter_saving_name), index=save_csv_index)

